In [6]:
# Import kamping library before starting the tutorial
import kamping

%load_ext autoreload
%autoreload 2

In [7]:
gene_graphs = kamping.create_graphs('../data/kgml_hsa', type='mixed', verbose=True, ignore_file=['hsa01100.xml'])


            Visit https://www.kegg.jp/kegg-bin/show_pathway?hsa00190 for pathway details.

            There are likely no edges in which to parse...
INFO:KeggGraph:Now parsing: path:hsa00220...
INFO:KeggGraph:Graph path:hsa00220 parsed successfully!
INFO:KeggGraph:Now parsing: path:hsa00230...
INFO:KeggGraph:Graph path:hsa00230 parsed successfully!
INFO:KeggGraph:Now parsing: path:hsa00232...
INFO:KeggGraph:Graph path:hsa00232 parsed successfully!
INFO:KeggGraph:Now parsing: path:hsa00240...
INFO:KeggGraph:Graph path:hsa00240 parsed successfully!
INFO:KeggGraph:Now parsing: path:hsa00250...
INFO:KeggGraph:Graph path:hsa00250 parsed successfully!
INFO:KeggGraph:Now parsing: path:hsa00260...
INFO:KeggGraph:Graph path:hsa00260 parsed successfully!
INFO:KeggGraph:Now parsing: path:hsa00270...
INFO:KeggGraph:Graph path:hsa00270 parsed successfully!
INFO:KeggGraph:Now parsing: path:hsa00280...
INFO:KeggGraph:Graph path:hsa00280 parsed successfully!
INFO:KeggGraph:Now parsing: path:hsa00290

In [8]:
gene_graph_00010 = [graph for graph in gene_graphs if graph.name == 'path:hsa00010'][0]
gene_graph_00010

KEGG Pathway: 
            [Title]: Glycolysis / Gluconeogenesis
            [Name]: path:hsa00010
            [Org]: hsa
            [Link]: https://www.kegg.jp/kegg-bin/show_pathway?hsa00010
            [Image]: https://www.kegg.jp/kegg/pathway/hsa/hsa00010.png
            [Link]: https://www.kegg.jp/kegg-bin/show_pathway?hsa00010
            Graph type: mixed 
            Number of Genes: 67
            Number of Compounds: 26
            Gene ID type : kegg
            Compound ID type : kegg
            Number of Nodes: 93
            Number of Edges: 279

In [9]:
converter = kamping.Converter('hsa', gene_target='uniprot', verbose=True)

In [10]:
for graph in gene_graphs:
    converter.convert(graph)

INFO:kamping.parser.convert:Conversion of path:hsa00010 complete!
INFO:kamping.parser.convert:Conversion of path:hsa00020 complete!
INFO:kamping.parser.convert:Conversion of path:hsa00030 complete!
INFO:kamping.parser.convert:Conversion of path:hsa00040 complete!
INFO:kamping.parser.convert:Conversion of path:hsa00051 complete!
INFO:kamping.parser.convert:Conversion of path:hsa00052 complete!
INFO:kamping.parser.convert:Conversion of path:hsa00053 complete!
INFO:kamping.parser.convert:Conversion of path:hsa00061 complete!
INFO:kamping.parser.convert:Conversion of path:hsa00062 complete!
INFO:kamping.parser.convert:Conversion of path:hsa00071 complete!
INFO:kamping.parser.convert:Conversion of path:hsa00100 complete!
INFO:kamping.parser.convert:Conversion of path:hsa00120 complete!
INFO:kamping.parser.convert:Conversion of path:hsa00130 complete!
INFO:kamping.parser.convert:Conversion of path:hsa00140 complete!
INFO:kamping.parser.convert:Conversion of path:hsa00220 complete!
INFO:kampi

In [15]:
import pandas as pd

# uncommented code below if run the first time
# save the mols to a file
# mols.to_pickle('data/mols.pkl')
# retrieve mol from file
mols = pd.read_pickle('../data/mols.pkl')
mol_embeddings = kamping.get_mol_embeddings_from_dataframe(mols, transformer='morgan')

NameError: name 'mols' is not defined

In [13]:
mol_embeddings

NameError: name 'mol_embeddings' is not defined

In [ ]:
protein_embeddings = kamping.get_uniprot_protein_embeddings(gene_graphs, '../data/embedding/protein_embedding.h5') 
protein_embeddings

In [ ]:
# combine protein embeddings and metabolite embeddings into one dictionary
embeddings = {**protein_embeddings, **mol_embeddings}
len(embeddings)

In [ ]:
import pandas as pd
string_data = pd.read_csv("../data/9606.protein.links.v12.0-translated.csv", usecols=['protein1', 'protein2', 'combined_score'])
string_data = string_data[string_data['combined_score'] > 400]
# add "up:" prefix to the protein id
string_data['protein1'] = 'up:' + string_data['protein1']
string_data['protein2'] = 'up:' + string_data['protein2']

In [ ]:
# use the df to create a list of tuples
additional_edges = list(string_data.itertuples(index=False, name=None))
additional_edges
# and add {'type': 'PPrel', 'subtype_name': 'additional', 'subtype_value': 'additional', 'entry1_type': 'protein', 'entry2_type': 'protein'} to each tuple
additional_edges = [(edge[0], edge[1], {'type': 'PPrel', 'subtype_name': 'additional', 'subtype_value': 'additional', 'entry1_type': 'protein', 'entry2_type': 'protein'}) for edge in additional_edges]
additional_edges

In [ ]:
pyg_one_graph, mapping = kamping.convert_to_single_pyg(gene_graphs, embeddings=protein_embeddings, additional_edges=additional_edges)

In [ ]:
pyg_graph, mapping = kamping.convert_to_single_pyg(gene_graphs, embeddings=embeddings)
data= pyg_graph
data, mapping

# You can save the data as pickle file for later use

In [ ]:
import pickle

# save the data as pickle file
with open('../data/pyg_graph.pkl', 'wb') as f:
    pickle.dump(data, f)

In [ ]:
import torch_geometric.transforms as T
transform = T.RandomLinkSplit(
    num_val=0.1,
    num_test=0.1,
    is_undirected=True,
    # disjoint_train_ratio=0.3, # TODO
    neg_sampling_ratio=1.0, # TODO
    add_negative_train_samples=False,
    edge_types=("gene", "to", "gene")
)
train_data, val_data, test_data = transform(data)

In [ ]:
val_data

In [ ]:
import torch

In [ ]:
from sklearn.metrics import roc_auc_score